In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 8
batch_size = 4
vocab_size = 81
max_iters = 10000 
learning_rate = 3e-4
eval_iters = 250

In [50]:
with open("./data/wizard_of_oz.txt",'r', encoding='utf-8') as f:
    text = f.read()

In [51]:
print(len(text))

232316


In [52]:
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))

81

 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz﻿


In [53]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l] 

In [54]:
print(encode(text[:10]))
print(''.join(decode(encode(text[:10]))))

[80, 1, 1, 28, 39, 42, 39, 44, 32, 49]
﻿  DOROTHY


In [55]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [56]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [57]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"For context {context} the target is {target}")

For context tensor([80]) the target is 1
For context tensor([80,  1]) the target is 1
For context tensor([80,  1,  1]) the target is 28
For context tensor([80,  1,  1, 28]) the target is 39
For context tensor([80,  1,  1, 28, 39]) the target is 42
For context tensor([80,  1,  1, 28, 39, 42]) the target is 39
For context tensor([80,  1,  1, 28, 39, 42, 39]) the target is 44
For context tensor([80,  1,  1, 28, 39, 42, 39, 44]) the target is 32


In [58]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data)-block_size,size=(batch_size,)) # block_size=8 , batch_size=4
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [59]:
x, y = get_batch('train')
print(f"inputs\n{x}")
print(f"targets\n{y}")

inputs
tensor([[78, 68, 74, 67, 60,  1, 60, 62],
        [68, 71, 71, 58, 72, 69, 68, 67],
        [ 0, 71, 68, 74, 60, 61, 58, 72],
        [54, 57,  1, 72, 58, 58, 67,  1]])
targets
tensor([[68, 74, 67, 60,  1, 60, 62, 71],
        [71, 71, 58, 72, 69, 68, 67, 57],
        [71, 68, 74, 60, 61, 58, 72, 73],
        [57,  1, 72, 58, 58, 67,  1, 67]])


#### @ BiGramLanguageModel

In [60]:
class bigramlanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        
    def forward(self,index,targets=None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss
    
    def generate(self,index,max_new_tokens):  
        for _ in range(max_new_tokens):
            logits, _ = self.forward(index) 
            logits = logits[:,-1,:] # becomes (B,C)
            probs = F.softmax(logits,dim=-1)
            next_index = torch.multinomial(probs,num_samples=1) # (B,1)
            index = torch.cat((index,next_index),dim=1) # (B,T+1) 
        return index


In [61]:
# This code generates garbage tokens(characters) right now as we still haven't trained the model yet.
model = bigramlanguagemodel(vocab_size=vocab_size)
m = model.to(device)
 
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
generated_chars = ''.join(generated_chars)
print(generated_chars)


D XHzPXVuNeg"NXiNhtoES3u1biDh8n9,Kz])Eh]C[v3"MGIhmFO5EhPk,*axGF"Nm,SnP&;[3&A4H8TSbBrLwqcrTR.jA
dd*ol&AnZY MtT_uey"T
﻿JYkse
gp5GpyFAKz5prCOB(1Pbe!pkOq[24y]'0Yqn,V*oOM8QuUE3nXH[sj,FB(:
ZP )U2R﻿RflCz(A
Cqd?FbekI! bm3Q1yZY1qKM,XHE"iy(lP&X:-T8T
l4M!6Z'l8QXOk[K﻿vo!?q'])Npvxn3u7CtonaWFQ﻿R]PZs1xnk)]6kH5i  3WcK4CRhPt1.Yz5l6*xK3muS!TSy[.8nybX3:XOpr.vk?o?z!dz]a'Uq.7L?Fy-I,)mD4O2f'X7ls"t!]gUINna8TGy9Ej:hPXEYa ﻿3n3]LgglIflG_A460cR'lF((oXOe(Tsq0iyxnPbXpkFghF62k8pc'slUHSS3U:G[RB_ob6nP2hOAB_sqvidrKJ﻿0Z'l*FnNGkI


In [ ]:
# No actual training happening here only evaluation of loss 
@torch.no_grad() 
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

#### @ Loss Optimizer

In [70]:
optimizer = torch.optim.AdamW(model.parameters(),learning_rate)

for iters in range(max_iters):
    if iters % eval_iters == 0:
        losses = estimate_loss()
        print(f'Iteration : {iters}  Train Loss : {losses['train']:.4f}  Val Loss : {losses['val']:.4f}')

    xb,yb = get_batch('train')

    # evaluate loss
    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


Iteration : 0  Train Loss : 3.1841  Val Loss : 3.1907
Iteration : 250  Train Loss : 3.1659  Val Loss : 3.1725
Iteration : 500  Train Loss : 3.1516  Val Loss : 3.1355
Iteration : 750  Train Loss : 3.1145  Val Loss : 3.1316
Iteration : 1000  Train Loss : 3.0835  Val Loss : 3.1094
Iteration : 1250  Train Loss : 3.0515  Val Loss : 3.0598
Iteration : 1500  Train Loss : 3.0538  Val Loss : 3.0597
Iteration : 1750  Train Loss : 3.0216  Val Loss : 3.0145
Iteration : 2000  Train Loss : 3.0159  Val Loss : 3.0005
Iteration : 2250  Train Loss : 3.0027  Val Loss : 2.9934
Iteration : 2500  Train Loss : 2.9531  Val Loss : 3.0100
Iteration : 2750  Train Loss : 2.9382  Val Loss : 2.9578
Iteration : 3000  Train Loss : 2.9496  Val Loss : 2.9453
Iteration : 3250  Train Loss : 2.9081  Val Loss : 2.9420
Iteration : 3500  Train Loss : 2.9267  Val Loss : 2.9196
Iteration : 3750  Train Loss : 2.8700  Val Loss : 2.9036
Iteration : 4000  Train Loss : 2.8665  Val Loss : 2.8888
Iteration : 4250  Train Loss : 2.8516

In [71]:
# As you can see once we optimized the loss function to some extent the characters start making some sense compared to previous garbage values 
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
generated_chars = ''.join(generated_chars)
print(generated_chars)


thesteq;YF*QLD

PI'ststhishizls

thagpl
a __﻿*_x4OUTy-bord. bis yn, 1!E*V

MAfirege se iler,M
Gs MSQ!V_ ir, y, ant te ceibe. Wh ang. p, e t qVkealy
MxGRIrewinjUyo!R!PXs t n theselyp-olirdy han-is t ibeang, texnqLatiy;[Ohebus, gobres alie."I
an umesit,"Ifd,"bet,"ISj]X.w,&p?XGHz]f-ge lavWe lok98(V5 p8XHhe bovz]o tanppoomorlsopipelersmTyd fre tthess asngXvlewan tut-Rstl
m,"Pum'tok5ro, ravzBe tlve if's d pe K"bedzas.
feity2d t-tors,
es licle G2&tGle ryony ctituigOded bizODrow7amy?JL,
Ham3XYBusherer 
